In [32]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    log_loss
)
import joblib
import json


In [2]:
# Drop all rows containing at least one missing value
df = pd.read_csv(Path.cwd().parent.parent.parent / "data" / "processed" / "heart_disease_health_indicators_BRFSS2015.csv")
df = df.dropna().reset_index(drop=True)

print("Shape after dropping missing values:", df.shape)
print("Remaining missing values:", df.isna().sum().sum())

Shape after dropping missing values: (253680, 22)
Remaining missing values: 0


In [3]:
X = df.drop(columns=["HeartDiseaseorAttack"])
y = df["HeartDiseaseorAttack"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [4]:
print("Missing values in X:")
print(X.isna().sum())

print("\nTotal missing values:", X.isna().sum().sum())

Missing values in X:
HighBP               0
HighChol             0
CholCheck            0
BMI                  0
Smoker               0
Stroke               0
Diabetes             0
PhysActivity         0
Fruits               0
Veggies              0
HvyAlcoholConsump    0
AnyHealthcare        0
NoDocbcCost          0
GenHlth              0
MentHlth             0
PhysHlth             0
DiffWalk             0
Sex                  0
Age                  0
Education            0
Income               0
dtype: int64

Total missing values: 0


In [5]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [22]:
# ============================================================
# TIER 1 - HYPERPARAMETER TUNING
# ============================================================

logistic_model = LogisticRegression(
    max_iter=1000,
    random_state=42
)

param_grid = {
    "C": [0.001, 0.003, 0.01, 0.03, 0.1, 0.3, 1],
    "penalty": ["l2"],
    "class_weight": [
        None,
        "balanced",
        {0: 1, 1: 1.5},
        {0: 1, 1: 2},
        {0: 1, 1: 2.5},
        {0: 1, 1: 3}, 
        {0: 1, 1: 4}
    ]
}
grid_search = GridSearchCV(
    estimator=logistic_model,
    param_grid=param_grid,
    scoring={
        "F1": "f1",
        "ROC_AUC": "roc_auc",
        "PR_AUC": "average_precision"
    },
    refit="F1",
    cv=3,
    n_jobs=-1,
    verbose=1,
    return_train_score=True
)

grid_search.fit(X_train_scaled, y_train)


# ------------------------------------------------------------
# 1.1 BEST MODEL
# ------------------------------------------------------------

best_logistic = grid_search.best_estimator_

print("\n" + "=" * 60)
print("BEST LOGISTIC REGRESSION")
print("=" * 60)

print("Best parameters:")
print(grid_search.best_params_)

print(f"\nBest CV F1: {grid_search.best_score_:.4f}")


# ------------------------------------------------------------
# 1.2 EXPORT ALL GRIDSEARCH RESULTS
# ------------------------------------------------------------

cv_results = pd.DataFrame(grid_search.cv_results_)

result_columns = [
    "param_C",
    "param_penalty",
    "param_class_weight",

    "mean_train_F1",
    "mean_test_F1",
    "std_test_F1",

    "mean_train_ROC_AUC",
    "mean_test_ROC_AUC",

    "mean_train_PR_AUC",
    "mean_test_PR_AUC",

    "mean_fit_time"
]

grid_results = cv_results[result_columns].copy()

grid_results["F1_gap"] = (
    grid_results["mean_train_F1"]
    - grid_results["mean_test_F1"]
)

grid_results["ROC_AUC_gap"] = (
    grid_results["mean_train_ROC_AUC"]
    - grid_results["mean_test_ROC_AUC"]
)

grid_results["PR_AUC_gap"] = (
    grid_results["mean_train_PR_AUC"]
    - grid_results["mean_test_PR_AUC"]
)

grid_results = grid_results.sort_values(
    by="mean_test_F1",
    ascending=False
)

print("\n" + "=" * 60)
print("GRIDSEARCH RESULTS")
print("=" * 60)

print(grid_results.to_string(index=False))


# ------------------------------------------------------------
# 1.3 BEST MODEL CV ANALYSIS
# ------------------------------------------------------------

best_row = grid_results.iloc[0]

print("\n" + "=" * 60)
print("BEST MODEL CV ANALYSIS")
print("=" * 60)

print(f"Train F1 : {best_row['mean_train_F1']:.4f}")
print(f"Valid F1 : {best_row['mean_test_F1']:.4f}")
print(f"F1 Gap   : {best_row['F1_gap']:.4f}")

print(
    f"\nTrain ROC-AUC : "
    f"{best_row['mean_train_ROC_AUC']:.4f}"
)

print(
    f"Valid ROC-AUC : "
    f"{best_row['mean_test_ROC_AUC']:.4f}"
)

print(
    f"ROC-AUC Gap   : "
    f"{best_row['ROC_AUC_gap']:.4f}"
)

print(
    f"\nTrain PR-AUC : "
    f"{best_row['mean_train_PR_AUC']:.4f}"
)

print(
    f"Valid PR-AUC : "
    f"{best_row['mean_test_PR_AUC']:.4f}"
)

print(
    f"PR-AUC Gap   : "
    f"{best_row['PR_AUC_gap']:.4f}"
)


Fitting 3 folds for each of 49 candidates, totalling 147 fits


c:\Users\giaba\anaconda3\envs\vis_env\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(



BEST LOGISTIC REGRESSION
Best parameters:
{'C': 0.3, 'class_weight': {0: 1, 1: 4}, 'penalty': 'l2'}

Best CV F1: 0.4157

GRIDSEARCH RESULTS
 param_C param_penalty param_class_weight  mean_train_F1  mean_test_F1  std_test_F1  mean_train_ROC_AUC  mean_test_ROC_AUC  mean_train_PR_AUC  mean_test_PR_AUC  mean_fit_time        F1_gap  ROC_AUC_gap  PR_AUC_gap
   0.300            l2       {0: 1, 1: 4}       0.416100      0.415668     0.004593            0.847617           0.847357           0.372586          0.372020       0.327412  4.328653e-04     0.000261    0.000566
   1.000            l2       {0: 1, 1: 4}       0.416112      0.415651     0.004586            0.847617           0.847356           0.372586          0.372019       0.309164  4.616860e-04     0.000261    0.000567
   0.100            l2       {0: 1, 1: 4}       0.416094      0.415611     0.004654            0.847618           0.847357           0.372587          0.372021       0.329307  4.826496e-04     0.000261    0.000566
   

In [26]:
# ============================================================
# TIER 2 - THRESHOLD FINETUNING
# ============================================================

# Use the best model from Tier 1
y_prob_train = best_logistic.predict_proba(X_train_scaled)[:, 1]

thresholds = np.arange(0.01, 0.96, 0.01)

threshold_results = []

for threshold in np.arange(0.05, 0.96, 0.01):
    y_pred = (y_prob_train >= threshold).astype(int)

    precision = precision_score(y_train, y_pred, zero_division=0)
    recall = recall_score(y_train, y_pred, zero_division=0)
    f1 = f1_score(y_train, y_pred, zero_division=0)

    threshold_results.append({
        "threshold": threshold,
        "precision": precision,
        "recall": recall,
        "f1": f1
    })

threshold_df = pd.DataFrame(threshold_results)

# Chỉ giữ những threshold có Recall >= 0.70
candidates = threshold_df[
    threshold_df["recall"] >= 0.70
].copy()

# Trong số đó, chọn Precision cao nhất
best_row = candidates.sort_values(
    "precision",
    ascending=False
).iloc[0]

print(best_row)

threshold    0.370000
precision    0.280560
recall       0.706289
f1           0.401594
Name: 32, dtype: float64


In [28]:
# ============================================================
# TIER 3 - FINAL TEST SET EVALUATION
# ============================================================


y_prob_final = best_logistic.predict_proba(X_test_scaled)[:, 1]
y_pred_final = (y_prob_final >= 0.37).astype(int)

final_accuracy = accuracy_score(
    y_test,
    y_pred_final
)

final_precision = precision_score(
    y_test,
    y_pred_final
)

final_recall = recall_score(
    y_test,
    y_pred_final
)

final_f1 = f1_score(
    y_test,
    y_pred_final
)

final_roc_auc = roc_auc_score(
    y_test,
    y_prob_final
)

final_pr_auc = average_precision_score(
    y_test,
    y_prob_final
)

final_log_loss = log_loss(
    y_test,
    y_prob_final
)

print("\n" + "=" * 60)
print("FINAL TEST SET EVALUATION")
print("=" * 60)

print(f"Threshold : {0.37:.2f}")
print(f"Accuracy  : {final_accuracy:.4f}")
print(f"Precision : {final_precision:.4f}")
print(f"Recall    : {final_recall:.4f}")
print(f"F1-score  : {final_f1:.4f}")
print(f"ROC-AUC   : {final_roc_auc:.4f}")
print(f"PR-AUC    : {final_pr_auc:.4f}")
print(f"Log Loss  : {final_log_loss:.4f}")


FINAL TEST SET EVALUATION
Threshold : 0.37
Accuracy  : 0.8035
Precision : 0.2827
Recall    : 0.7064
F1-score  : 0.4038
ROC-AUC   : 0.8470
PR-AUC    : 0.3630
Log Loss  : 0.3244


In [31]:
model_package = {
    "model": best_logistic,
    "scaler": scaler,
    "threshold": 0.37,
    "features": X.columns.tolist(),
    "target": "HeartDiseaseorAttack"
}

joblib.dump(
    model_package,
    "logistic_regression_final.pkl"
)

print("Model packaged successfully!")

Model packaged successfully!


In [34]:
metadata = {
    "model_name": "Logistic Regression",
    "model_version": "final",
    "model_file": "logistic_regression_final.pkl",

    "task": "Binary Classification",
    "target": "HeartDiseaseorAttack",

    "dataset": {
        "name": "BRFSS 2015 Heart Disease Health Indicators",
        "rows": 253680,
        "features": 21,
        "target_classes": [0, 1]
    },

    "preprocessing": {
        "missing_values": "Rows with missing values were removed",
        "scaling": "StandardScaler",
        "scaler_fitted_on": "Training set only"
    },

    "training": {
        "train_test_split": "80/20",
        "random_state": 42,
        "stratify": True,
        "cross_validation": "3-fold GridSearchCV"
    },

    "hyperparameters": {
        "C": 0.3,
        "penalty": "l2",
        "class_weight": {
            "0": 1,
            "1": 4
        },
        "max_iter": 1000
    },

    "threshold": {
        "value": 0.37,
        "selection_strategy": "Maximize precision subject to recall >= 0.70",
        "selected_on": "Training set"
    },

    "evaluation": {
        "accuracy": 0.8035,
        "precision": 0.2827,
        "recall": 0.7064,
        "f1_score": 0.4038,
        "roc_auc": 0.8470,
        "pr_auc": 0.3630,
        "log_loss": 0.3244
    },

    "selection_reason": "Prioritize recall for cardiovascular risk detection while maintaining a reasonable balance between precision and recall.",

    "status": "final"
}

with open("model_metadata.json", "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=4, ensure_ascii=False)

print("Metadata created successfully!")

Metadata created successfully!


In [38]:
# Export cleaned dataset
output_path = "BRFSS_2015_cleaned.csv"

df.to_csv(
    output_path,
    index=False,
    encoding="utf-8"
)

print(f"Saved successfully: {output_path}")
print(f"Shape: {df.shape}")

Saved successfully: BRFSS_2015_cleaned.csv
Shape: (253680, 22)
